# Soil Image Classification with a CNN (Beginner Notebook)

This notebook walks through the image-model workflow used in **Fieldwise Smart Agriculture Assistant**. It classifies a soil photo into one of the dataset's visual soil categories. It does **not** measure fertility, pH, NPK, or recommend a crop by itself.

**Model idea:** start with MobileNetV2, a neural network already trained on millions of general images, replace its last layer with our soil classes, then train that new classifier on our soil photos. This is called **transfer learning**.

Run the cells from top to bottom. Training can take time, especially on a CPU. The project already has trained weights in `models/soil_cnn.keras`; rerunning the training cell will replace that file with the newly trained model.

## 1. What the training process does

1. Read images from one folder per soil class.
2. Make an **80% training / 10% validation / 10% test** split. The test set stays untouched until final evaluation.
3. Resize each photo to 224 × 224 pixels.
4. Apply small random flips, rotations, and zooms to **training images only**. This helps the model handle slightly different photos.
5. Use MobileNetV2 as a frozen feature extractor and train a small output layer for the soil classes.
6. Measure final performance on the held-out test images and save the model.

The app uses separate models for image class, crop suggestion, and fertility level; the CNN's visual class is not treated as a fertility measurement.

## 2. Check the project folders

The original dataset should be in `data/soil_images/`, with one folder per class. Example:

```text
data/soil_images/
  Arid Soil/
  Black Soil/
  Laterite Soil/
  Red Soil/
  ...
```

The dataset referenced in the project is licensed **CC BY-NC 4.0**; check its source repository for attribution and usage restrictions. The code below creates a separate processed-image cache and does not overwrite the original photos.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# This works whether Jupyter starts in the project folder or notebooks/.
CURRENT_FOLDER = Path.cwd()
PROJECT_FOLDER = CURRENT_FOLDER.parent if CURRENT_FOLDER.name == "notebooks" else CURRENT_FOLDER
SOURCE_FOLDER = PROJECT_FOLDER / "data" / "soil_images"
CACHE_FOLDER = PROJECT_FOLDER / "data" / "processed_soil_images"
MODEL_FOLDER = PROJECT_FOLDER / "models"
ARTIFACT_FOLDER = PROJECT_FOLDER / "artifacts"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 24
SEED = 42
SUPPORTED_FORMATS = {".jpg", ".jpeg", ".png", ".webp"}

tf.keras.utils.set_random_seed(SEED)

print("Project folder:", PROJECT_FOLDER)
print("Image folder exists:", SOURCE_FOLDER.exists())
if not SOURCE_FOLDER.exists():
    raise FileNotFoundError("Download or place the class folders in data/soil_images first.")

## 3. Prepare and label the photos

A folder name becomes the class label. For example, every photo inside `Arid Soil/` gets the label `Arid Soil`.

We verify images, skip exact duplicate files, and convert each valid image to ordinary RGB JPEG. This cache helps TensorFlow read formats such as WebP. The original photos stay unchanged.

In [ ]:
class_names = sorted(folder.name for folder in SOURCE_FOLDER.iterdir() if folder.is_dir())
if len(class_names) < 2:
    raise ValueError("Expected at least two class folders inside data/soil_images.")

image_paths = []
image_labels = []
seen_hashes = set()

for class_number, class_name in enumerate(class_names):
    class_folder = SOURCE_FOLDER / class_name
    for photo_path in class_folder.rglob("*"):
        if photo_path.suffix.lower() not in SUPPORTED_FORMATS:
            continue
        try:
            # Ignore byte-for-byte duplicates so copies cannot leak across splits.
            file_hash = hashlib.sha256(photo_path.read_bytes()).hexdigest()
            if file_hash in seen_hashes:
                continue
            with Image.open(photo_path) as photo:
                photo.verify()
            seen_hashes.add(file_hash)

            cached_photo = CACHE_FOLDER / class_name / f"{file_hash[:16]}.jpg"
            cached_photo.parent.mkdir(parents=True, exist_ok=True)
            if not cached_photo.exists():
                with Image.open(photo_path) as photo:
                    photo.convert("RGB").save(cached_photo, format="JPEG", quality=95)

            image_paths.append(str(cached_photo))
            image_labels.append(class_number)
        except (UnidentifiedImageError, OSError):
            print("Skipping unreadable file:", photo_path)

image_paths = np.array(image_paths)
image_labels = np.array(image_labels)

if len(image_paths) == 0:
    raise ValueError("No readable images found. Check the class folders and image files.")

print("Classes:", class_names)
print("Total usable photos:", len(image_paths))

## 4. Check how many photos each class has

Balanced class counts help learning. If one soil class has many more photos than another, a model might focus on the larger class. We will also use class weights later to give underrepresented classes extra importance during training.

In [ ]:
class_counts = {
    class_name: int(np.sum(image_labels == class_number))
    for class_number, class_name in enumerate(class_names)
}

for class_name, count in class_counts.items():
    print(f"{class_name}: {count} photos")

plt.figure(figsize=(10, 4))
plt.bar(class_counts.keys(), class_counts.values(), color="#5a9b60")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Number of photos")
plt.title("Photos in each soil class")
plt.tight_layout()
plt.show()

## 5. Split into training, validation, and test data

- **Training set:** teaches the model.
- **Validation set:** helps choose when to stop training.
- **Test set:** a final, once-only check on photos the model did not train on.

`stratify` keeps approximately the same class balance in each split.

In [ ]:
train_paths, remaining_paths, train_labels, remaining_labels = train_test_split(
    image_paths,
    image_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=image_labels,
)

validation_paths, test_paths, validation_labels, test_labels = train_test_split(
    remaining_paths,
    remaining_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=remaining_labels,
)

print("Training photos:", len(train_paths))
print("Validation photos:", len(validation_paths))
print("Test photos:", len(test_paths))

## 6. Make TensorFlow image batches

A batch is a small group of images processed together. The loader reads an image, decodes it, resizes it to 224 × 224, and groups it into batches. Shuffling is used for training only.

In [ ]:
def read_and_resize(photo_path, label):
    file_bytes = tf.io.read_file(photo_path)
    image = tf.io.decode_jpeg(file_bytes, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, label


def make_dataset(photo_paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((photo_paths, labels))
    if training:
        dataset = dataset.shuffle(len(photo_paths), seed=SEED)
    dataset = dataset.map(read_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_paths, train_labels, training=True)
validation_dataset = make_dataset(validation_paths, validation_labels)
test_dataset = make_dataset(test_paths, test_labels)

print("Datasets are ready.")

## 7. Look at a small batch of training photos

This is a quick visual check that labels and images match before training.

In [ ]:
for image_batch, label_batch in train_dataset.take(1):
    plt.figure(figsize=(10, 8))
    for index in range(min(6, len(image_batch))):
        plt.subplot(2, 3, index + 1)
        plt.imshow(image_batch[index].numpy().astype("uint8"))
        plt.title(class_names[int(label_batch[index])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Build the CNN using transfer learning

MobileNetV2 has already learned general image patterns such as edges, shapes, and textures. We freeze it so its existing knowledge is not changed at first. A small new output layer learns which patterns correspond to our soil classes.

The augmentation layer makes slight random changes during training. It is inside the model, so it is inactive when we evaluate or predict.

In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.08),
    ],
    name="training_augmentation",
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # Freeze the pre-trained feature extractor.

image_input = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = augmentation(image_input)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
class_output = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(image_input, class_output)
model.summary()

## 9. Configure training

- **Adam** adjusts model weights to reduce errors.
- **Sparse categorical cross-entropy** is suitable because each image has one integer class label.
- **Accuracy** is easy to understand, but later we also inspect per-class precision/recall and Macro F1.
- **Class weights** help avoid ignoring classes with fewer examples.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

unique_classes = np.unique(train_labels)
weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=train_labels,
)
class_weights = {
    int(class_number): float(weight)
    for class_number, weight in zip(unique_classes, weights)
}
print("Class weights:", class_weights)

MODEL_FOLDER.mkdir(parents=True, exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_FOLDER / "soil_cnn.keras"),
        monitor="val_loss",
        save_best_only=True,
    ),
]

## 10. Train the model

One **epoch** means the model has processed all training batches once. We allow up to 10 epochs. Early stopping ends training if validation loss stops improving, and the checkpoint keeps the best validation version.

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=callbacks,
    class_weight=class_weights,
    shuffle=False,
)

## 11. Review the learning curves

Training accuracy shows fit to the training photos. Validation accuracy estimates performance on held-out validation photos. If training accuracy keeps rising while validation performance gets worse, the model may be overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 12. Evaluate once on the held-out test photos

Do not train on the test set. Accuracy is the fraction classified correctly. **Macro F1** calculates an F1 score for each class and averages them equally, so small classes count too.

In [ ]:
# Load the best checkpoint saved during training.
best_model = tf.keras.models.load_model(MODEL_FOLDER / "soil_cnn.keras")
probabilities = best_model.predict(test_dataset, verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)

print("Test accuracy:", round(accuracy_score(test_labels, predicted_labels), 4))
print("Test Macro F1:", round(f1_score(test_labels, predicted_labels, average="macro", zero_division=0), 4))
print("\nPer-class report:")
print(
    classification_report(
        test_labels,
        predicted_labels,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
    )
)

In [ ]:
matrix = confusion_matrix(test_labels, predicted_labels, labels=np.arange(len(class_names)))
plt.figure(figsize=(8, 7))
plt.imshow(matrix, cmap="Blues")
plt.colorbar(label="Number of photos")
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.xlabel("Model prediction")
plt.ylabel("Correct class")
plt.title("Confusion matrix on test photos")
for row in range(len(class_names)):
    for column in range(len(class_names)):
        plt.text(column, row, matrix[row, column], ha="center", va="center")
plt.tight_layout()
plt.show()

## 13. Save the class names and try one photo

The app needs both the model file and the class-name file so it can translate the model's output number into a readable soil class.

In [ ]:
ARTIFACT_FOLDER.mkdir(parents=True, exist_ok=True)
class_names_path = ARTIFACT_FOLDER / "soil_class_names.json"
class_names_path.write_text(json.dumps(class_names, indent=2), encoding="utf-8")

# Try one unseen test photo.
sample_path = test_paths[0]
sample_bytes = tf.io.read_file(sample_path)
sample_image = tf.io.decode_jpeg(sample_bytes, channels=3)
sample_image = tf.image.resize(sample_image, IMAGE_SIZE)
sample_batch = tf.expand_dims(sample_image, axis=0)
sample_scores = best_model.predict(sample_batch, verbose=0)[0]
answer_number = int(np.argmax(sample_scores))

print("Photo:", sample_path)
print("Predicted soil class:", class_names[answer_number])
print("Model score:", round(float(sample_scores[answer_number]), 3))
print("Saved CNN:", MODEL_FOLDER / "soil_cnn.keras")
print("Saved class names:", class_names_path)

## Important interpretation

This model learns visual patterns from a limited, curated image dataset. A photograph cannot reveal measured soil nutrients or prove that soil is fertile/infertile. A high test score does not guarantee performance on a new farm, camera, lighting condition, or region. Use the visual class only as a learning aid; use a laboratory soil test and local agricultural specialist for real soil-management decisions.

**In this project:** the CNN classifies soil images. The separate tabular fertility model predicts the dataset's fertility category, and the separate crop model suggests crops from NPK/weather inputs. They are distinct workflows.